# Model C, C1 (Oct 30, 2026) — EXPLORATORY

"Start late, cross in human time, sit at the wall, switch by the frozen M2' hazard". `results/model_b_log.md`, entry 2: the spec, the yardstick and the implementation choices were declared there before any C1 result. Code: `src/slider/model_c1.py`, `sim.py` motion / preset "C1", `tests/test_c1.py`.

**Spec (as given):**

> TASK: Model C, version C1 (EXPLORATORY): "start late, cross in human time, sit at the wall,
> switch by the frozen M2' hazard". Fits on G-train only; simulations on the usual L3/L5
> designs. Never load Ring/CT. New code path only: all old presets (L1, final, L1_M3, L0,
> APEM, B1) must stay bit-identical (fingerprint test).
> Log C1 as a new entry in results/model_b_log.md, with its exact spec AND its yardstick,
> declared now, before any C1 result:
> - primary   = targets 1-4 (success, consensus time, % fraught, committed split);
> - secondary = targets 5-7;
> - descriptive only = targets 8-10 (10a matches by construction, so it is not a check).
>
> 0. DATA (G-train, per interface: lab v09, old CL, CC; human seats as in P17)
>    (a) Wall-to-wall trips: from the moment a player leaves the wall (|s| = 1) to its first
>        arrival at the OPPOSITE wall, without touching the starting wall again in between.
>        Keep the full duration, pauses and wobbles included.
>    (b) First trips: from the player's first move to its first arrival at any wall. Record
>        the duration and which wall.
>    (c) Descriptive (decision D2): the share of reversals (m = 2 detector) that start
>        mid-slider (|s| < 0.9) vs. in the end band.
>    Report counts, medians, quartiles, and the share of censored trips (round ended first;
>    excluded). Save the durations to results/tables/pC_durations.parquet.
>
> 1. SIMULATOR, preset "C1" (per tick, as in sim.py):
>    - Start: each player waits at s = 0 with p_start = lam_start*dt/(1 + lam_start*dt) per
>      tick (P17 lam_start per interface). Then a FIRST TRIP to a coin-flip wall, with
>      duration D drawn from (b) for that interface.
>    - Trips and crossings move on the grid at a constant pace: the n steps are spread evenly
>      over max(n, round(D/dt)) ticks. No stops, no reversals during a trip.
>    - At the wall: the player sits. The frozen M2' switch hazard is drawn per 0.25 s bin
>      exactly as in F2 (bins from the first move; features from the state before the bin;
>      time features at the bin start; no hazard in bins that start during a trip; at most
>      one switch per bin). The hazard is ON only while sitting at the wall.
>    - Switch -> a committed CROSSING to the opposite wall, with duration drawn from (a).
>      tau_r resets at the crossing's first step; tau_e follows the end-band rule as before.
>    - Old CL: grid 0.02, same logic. Round end: the consensus rule (|sum s|/7 >= 0.99) or
>      the time limit.
>    - Draw durations with the simulator's own seeded streams.
>
> 2. BUILD CHECKS (G6train design, R = 2, seed 573)
>    - Simulated trip and crossing durations reproduce (a) and (b): quantiles within the grid
>      resolution.
>    - lam_start re-estimated with the P17 code: within its CI.
>    - Hazard calibration: on simulated at-wall bins, observed switches vs. the sum of the
>      hazard (overall and by A decile) within +/-3%.
>    - The round_end agreement test and the old-preset fingerprint test pass.
>    - If anything fails: diagnose and propose; don't change numbers.
>
> 3. RUN SCRIPTS: register "C1" (seed 7001) in scripts/p24_main.py and p24_checks.py.
>    Report C1 next to final, L1 and B1, labeled EXPLORATORY, with no triggers, plus a
>    "C yardstick" summary: misses among primary / secondary separately; 10a marked "by
>    construction". Smoke-test with R = 2, delete the outputs, restore p24_checks.csv.
>
> Commit. Report steps 0-3 and give me the R = 200 command. Then stop.

## Choices (declared in the log before any result)

- **(a) Wall-to-wall trips.** Every departure from a wall (|s| from 1 to < 1) is one of three: arrived (it next touches the opposite wall; duration = arrival row - departure row), returned (it touches the starting wall first) or censored (the round ends first).
- **(b) First trips.** From the first own step to the first row at either wall; censored if no wall is reached.
- **Simulator pace.** A trip of D seconds takes L = max(n, round(D / dt) + 1) ticks, counting the first step's tick, so the simulated logs measure round(D / dt) dt. In lab v09, durations are pooled over rounds of speed 0.5, 1 and 2 units/s; a short D in a slow round runs at full pace.
- **At-risk bins.** The estimator row before the bin start shows |s| = 1, with no open reversal.
- **Build-check tolerance.** Quantile tolerance = the interface's largest round dt. Hazard calibration = complete at-risk bins (tau + 0.35 s <= round end), from the simulator trace.

In [1]:
import time

import numpy as np
import pandas as pd

from slider import config, design, model_c1, motion, sim
from slider import slider_logs as sl

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)

runs, table = sl.load_runs("behavior")                  # roles={"train"}: G-train only, no G-val, Ring or CT
assert set(table["role"]) == {"train"}
trips, revs = model_c1.tables(runs)
trips.to_parquet(config.PC_DURATIONS, index=False)
human = model_c1.summary(trips)

## 0 (a), (b). First trips and wall-to-wall trips (s): counts, outcomes, quantiles of the arrived trips

In [2]:
human.round(3)

,interface,kind,n,arrived,returned,censored,censored_share,q10,q25,median,q75,q90
0,lab_v09,first,609,609,0,0,0.000,0.900,0.940,1.460,3.450,7.512
1,old_cl,first,84,79,0,5,0.060,4.396,5.270,7.790,16.530,29.722
2,cc,first,930,930,0,0,0.000,0.755,0.765,1.199,3.470,7.545
3,lab_v09,cross,2344,1218,1123,3,0.001,2.200,2.203,2.400,4.600,9.700
4,old_cl,cross,383,72,301,10,0.026,10.200,10.300,10.600,16.948,25.890
5,cc,cross,2614,1640,961,13,0.005,1.056,1.093,1.122,2.385,4.810


In [3]:
first = trips[(trips["kind"] == "first") & (trips["outcome"] == "arrived")]
first.groupby("interface")["wall"].value_counts(normalize=True).unstack().round(3)

wall,-1.0,1.0
interface,,
cc,0.475,0.525
lab_v09,0.471,0.529
old_cl,0.481,0.519


## 0 (c). D2 (descriptive): where reversals start (m = 2 detector; position just before the first step)

In [4]:
d2 = revs.groupby("interface")["end_band"].agg(reversals="size", end_band="sum")
d2["mid_slider"] = d2["reversals"] - d2["end_band"]
d2["end_band_share"] = d2["end_band"] / d2["reversals"]
d2["mid_slider_share"] = d2["mid_slider"] / d2["reversals"]
d2.loc["pooled"] = d2.sum()
d2.loc["pooled", ["end_band_share", "mid_slider_share"]] = d2.loc["pooled", ["end_band", "mid_slider"]] / d2.loc["pooled", "reversals"]
d2.loc[motion.INTERFACES + ["pooled"]].round(3)

,reversals,end_band,mid_slider,end_band_share,mid_slider_share
interface,,,,,
lab_v09,2893.0,1864.0,1029.0,0.644,0.356
old_cl,363.0,188.0,175.0,0.518,0.482
cc,2871.0,2080.0,791.0,0.724,0.276
pooled,6127.0,4132.0,1995.0,NaN,NaN


**How to read it.** (a) and (b) are the human durations C1 draws from: first trips (0 to a wall) and wall-to-wall crossings, pauses and wobbles included. "returned" departures (the player steps off the wall and comes back to it) are not trips. Censored trips are excluded. (c) is the share of reversals that start in the end band vs. mid-slider; C1 has no mid-slider reversals at all.

## 2. Build checks: C1 on G6train, R = 2, seed 573 (the simulator trace kept per round)

In [5]:
G6 = design.design("G6train")
model = sim.preset("C1")
streams = np.random.SeedSequence(573).spawn(len(G6))           # design.simulate_rounds' streams
start = time.time()
rounds, bins = [], []
for row, stream in zip(G6.itertuples(), streams):
    sim_seed, log_seed = stream.spawn(2)
    A = row.graph[np.ix_(row.seat_node, row.seat_node)]
    trace = []
    out = sim.simulate(A, row.T, row.interface, R=2, seed=sim_seed, speed=row.speed, trace=trace, **model)
    rounds.append((out, np.random.default_rng(log_seed)))
    b = pd.concat([d for d in trace if "tau" in d], ignore_index=True)
    fires = [d for d in trace if "fire" in d]
    b["fired"] = False
    if fires:
        f = pd.concat(fires, ignore_index=True)
        for x in f.itertuples():                                  # the bin the switch is dated in
            k = b.index[(b["rep"] == x.rep) & (b["player"] == x.player) & (b["tau"] <= x.fire + 1e-6)][-1]
            b.loc[k, "fired"] = True
    end = np.array([o["end_time"] for o in out])
    b = b[b["risk"] & (b["tau"] + sim.BIN + 0.1 <= end[b["rep"]])]
    bins.append(b[["rep", "player", "tau", "h", "A", "fired"]].assign(interface=row.interface, round_id=row.round_id))
bins = pd.concat(bins, ignore_index=True)
check = design.simulate_rounds(G6.head(3), "C1", R=2, seed=573)    # the traced rounds are design.simulate_rounds'
assert all(a["events"].equals(b["events"]) for (x, _), (y, _) in zip(rounds[:3], check) for a, b in zip(x, y))
sim_runs = [run for r in range(2) for run in design.copy_runs(G6, rounds, r, 573)]
print(len(sim_runs), "rounds |", f"{(time.time() - start) / 60:.1f} min")

488 rounds | 7.3 min


### Durations: simulated vs. human quantiles (tolerance = the interface's largest round dt)

In [6]:
sim_trips, _ = model_c1.tables(sim_runs)
simulated = model_c1.summary(sim_trips)
tol = G6.groupby("interface")["dt"].max()
q = ["q10", "q25", "median", "q75", "q90"]
dur = human[["interface", "kind"] + q].merge(simulated[["interface", "kind", "arrived", "returned", "censored"] + q],
                                             on=["interface", "kind"], suffixes=("_human", "_sim"))
diff = np.column_stack([dur[f"{c}_sim"] - dur[f"{c}_human"] for c in q])
dur["max_abs_diff"] = np.abs(diff).max(axis=1)
dur["tolerance"] = dur["interface"].map(tol).astype(float)
dur["pass"] = dur["max_abs_diff"] <= dur["tolerance"] + 1e-9
dur.round(3)

,interface,kind,q10_human,q25_human,median_human,q75_human,q90_human,arrived,returned,censored,q10_sim,q25_sim,median_sim,q75_sim,q90_sim,max_abs_diff,tolerance,pass
0,lab_v09,first,0.900,0.940,1.460,3.450,7.512,1218,0,0,0.900,0.900,1.700,3.400,7.500,0.240,0.200,False
1,old_cl,first,4.396,5.270,7.790,16.530,29.722,168,0,0,4.900,5.300,7.800,16.300,33.110,3.388,0.100,False
2,cc,first,0.755,0.765,1.199,3.470,7.545,2030,0,0,0.748,0.759,1.189,3.434,7.268,0.277,0.034,False
3,lab_v09,cross,2.200,2.203,2.400,4.600,9.700,11308,0,85,2.200,2.200,2.400,4.600,9.700,0.002,0.200,True
4,old_cl,cross,10.200,10.300,10.600,16.948,25.890,1395,0,73,10.200,10.300,10.600,17.400,27.600,1.710,0.100,False
5,cc,cross,1.056,1.093,1.122,2.385,4.810,11945,0,58,1.055,1.089,1.122,2.410,4.753,0.057,0.034,False


### lam_start re-estimated with the P17 code (motion.motion_table: first moves / pre-first-move seconds)

In [7]:
seats_sim, _ = motion.motion_table(sim_runs)
p17 = pd.read_csv(config.P17_RATES)
lam = seats_sim.groupby("interface")[["moved", "pre_s"]].sum()
ls = p17[p17["rate"] == "lam_start"].set_index("interface").loc[motion.INTERFACES, ["estimate", "ci_low", "ci_high"]]
ls["re_estimate"] = lam["moved"] / lam["pre_s"]
ls["pass"] = (ls["re_estimate"] >= ls["ci_low"]) & (ls["re_estimate"] <= ls["ci_high"])
ls.round(4)

,estimate,ci_low,ci_high,re_estimate,pass
interface,,,,,
lab_v09,0.3171,0.2675,0.3714,0.3006,True
old_cl,0.0800,0.0563,0.1378,0.0797,True
cc,0.2037,0.1834,0.2280,0.2058,True


### Hazard calibration: at-risk at-wall bins, observed switches vs. the sum of h (+/- 3%)

In [8]:
def calib(d):
    exp, obs = d["h"].sum(), d["fired"].sum()
    se = np.sqrt((d["h"] * (1 - d["h"])).sum())
    return pd.Series({"bins": len(d), "expected": exp, "observed": obs, "rel_diff": obs / exp - 1, "rel_se": se / exp,
                      "pass": abs(obs / exp - 1) <= 0.03})

bins["A_decile"] = pd.qcut(bins["A"].rank(method="first"), 10, labels=range(1, 11))
overall = pd.concat([bins.groupby("interface")[["h", "fired"]].apply(calib).loc[motion.INTERFACES],
                     calib(bins).to_frame("pooled").T])
deciles = bins.groupby("A_decile", observed=True)[["h", "fired"]].apply(calib)
deciles["A_range"] = bins.groupby("A_decile", observed=True)["A"].agg(lambda a: f"[{a.min():.2f}, {a.max():.2f}]")
overall.round(4)

,bins,expected,observed,rel_diff,rel_se,pass
lab_v09,555224,11369.315093,11387,0.001555,0.009177,True
old_cl,83606,1475.753769,1467,-0.005932,0.025621,True
cc,529598,11961.421229,11995,0.002807,0.008936,True
pooled,1168428,24806.490091,24849,0.001714,0.006211,True


In [9]:
deciles.round(4)

,bins,expected,observed,rel_diff,rel_se,pass,A_range
A_decile,,,,,,,
1,116843,5767.5500,5820,0.0091,0.0127,True,"[0.00, 0.06]"
2,116843,5017.2335,5149,0.0263,0.0137,True,"[0.06, 0.17]"
3,116843,3915.6286,3915,-0.0002,0.0156,True,"[0.17, 0.30]"
4,116842,2979.6507,2876,-0.0348,0.0180,False,"[0.30, 0.40]"
5,116843,2310.4679,2289,-0.0093,0.0205,True,"[0.40, 0.50]"
6,116843,1827.3804,1790,-0.0205,0.0231,True,"[0.50, 0.63]"
7,116842,1202.6192,1249,0.0386,0.0286,False,"[0.63, 0.80]"
8,116843,665.0832,659,-0.0091,0.0386,True,"[0.80, 1.00]"
9,116843,522.6829,496,-0.0510,0.0436,False,"[1.00, 1.00]"


**How to read it.** C1 was simulated on every G-train round (2 copies, seed 573).
- **Durations:** the step-0 code measures the simulated logs, and the quantiles are compared with the human ones. PASS = every quantile is within one tick of the interface.
- **lam_start:** the P17 estimate on the simulated runs, against its human CI.
- **Calibration:** every complete at-risk bin (at the wall) traced in the simulator; observed switches vs. the sum of h, overall and by decile of A (deciles of the at-risk bins, pooled over interfaces). `rel_se` is the binomial SE of the relative difference.

## Diagnosis of the misses (no number changed; proposals in the log)

**Durations.** Two sources, both computed here without a new simulation:
- **Resampling noise.** The simulation draws n_sim durations from a pool of n_human values. The 95% range of (draw quantile - human quantile), over 1,000 sets of n_sim draws, is often wider than one tick: at the tails, and for the small old CL pools.
- **The full-pace floor.** A duration shorter than (n - 1) dt of the round it is drawn into runs at full pace. "pace shift" = the quantile of max(n - 1, round(D / dt)) dt over every (human trip's dt, pool D) pair, minus the human quantile. It is a deterministic expectation, approximate for crossings, whose rounds differ between sim and humans.

"inside noise + pace" = every quantile difference lies in the resampling range widened by the pace shift.

In [10]:
Q = [0.1, 0.25, 0.5, 0.75, 0.9]
N = {"lab_v09": 10, "old_cl": 50, "cc": 10}
arrived = trips[trips["outcome"] == "arrived"]
rng = np.random.default_rng(config.SEED)
rows = []
for r in dur.itertuples():
    x = arrived[(arrived["interface"] == r.interface) & (arrived["kind"] == r.kind)]
    D, dt = x["duration"].to_numpy(), x["dt"].to_numpy()
    n = N[r.interface] * (1 if r.kind == "first" else 2)
    hq = np.quantile(D, Q)
    pace = np.quantile((np.maximum(n - 1, np.round(D[None, :] / dt[:, None])) * dt[:, None]).ravel(), Q) - hq
    boot = np.array([np.quantile(rng.choice(D, r.arrived), Q) for _ in range(1000)]) - hq
    lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
    got = np.array([getattr(r, f"{c}_sim") - getattr(r, f"{c}_human") for c in q])
    inside = (got >= lo + np.minimum(pace, 0) - 1e-9) & (got <= hi + np.maximum(pace, 0) + 1e-9)
    rows.append({"interface": r.interface, "kind": r.kind, "n_human": len(D), "n_sim": r.arrived,
                 "below_full_pace": (D[None, :] < (n - 1) * dt[:, None] - 1e-9).mean(),
                 **{f"{c} sim-human": g for c, g in zip(q, got)}, **{f"{c} pace shift": p for c, p in zip(q, pace)},
                 **{f"{c} noise 95%": f"[{a:.3f}, {b:.3f}]" for c, a, b in zip(q, lo, hi)},
                 "inside noise + pace": bool(inside.all())})
pd.DataFrame(rows).set_index(["interface", "kind"]).round(3).T

interface,lab_v09,old_cl,cc,lab_v09,old_cl,cc
kind,first,first,first,cross,cross,cross
n_human,609,79,930,1218,72,1640
n_sim,1218,168,2030,11308,1395,11945
below_full_pace,0.095,0.19,0.012,0.095,0.014,0.002
q10 sim-human,0.0,0.504,-0.007,0.0,-0.0,-0.001
q25 sim-human,-0.04,0.03,-0.006,-0.002,0.0,-0.004
median sim-human,0.24,0.01,-0.01,0.0,0.0,0.0
q75 sim-human,-0.05,-0.23,-0.036,-0.0,0.452,0.025
q90 sim-human,-0.012,3.388,-0.277,-0.0,1.71,-0.057
q10 pace shift,0.0,0.504,-0.007,0.0,-0.0,0.0


**Hazard calibration by decile.** z = rel_diff / rel_se, and the chi-square over the 10 deciles:

In [11]:
from scipy import stats
z = deciles["rel_diff"] / deciles["rel_se"]
print("z by decile:", z.round(2).tolist())
print(f"chi-square {np.sum(z ** 2):.2f} on 10 df, p = {stats.chi2.sf(np.sum(z ** 2), 10):.3f}; "
      f"deciles with rel_se > 3% (the +/- 3% band is inside 1 SE): {int((deciles['rel_se'] > 0.03).sum())}")

z by decile: [0.71, 1.91, -0.01, -1.93, -0.45, -0.88, 1.35, -0.24, -1.17, 0.32]
chi-square 12.24 on 10 df, p = 0.269; deciles with rel_se > 3% (the +/- 3% band is inside 1 SE): 3
